In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    print("PASS: Application opened")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")
    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    print("PASS: Login successful")

    wait.until(EC.visibility_of_element_located((By.XPATH, "//section[@aria-label='Pharmacist dashboard']")))
    print("PASS: Dashboard opened")

    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Customers')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='Customers']")))
    print("PASS: Customers opened")

    # Search an existing customer: reuse the first listed name (no data created)
    wait.until(lambda d: d.find_elements(By.XPATH, "//span[contains(@class, 'cust-row-name')]"
        " | //*[text()='No customers found']"))
    time.sleep(1)
    names = [el.text.strip() for el in driver.find_elements(By.XPATH, "//span[contains(@class, 'cust-row-name')]") if el.text.strip()]
    if not names:
        print("SKIP: No existing customers for search/details steps")
    else:
        search = driver.find_element(By.XPATH, "//input[@aria-label='Search customers by name, phone or email']")
        search.clear()
        search.send_keys(names[0])
        time.sleep(2)
        assert names[0] in driver.find_element(By.TAG_NAME, "body").text
        print(f"PASS: Customer search completed ({names[0]})")
        driver.find_element(By.XPATH, "(//span[contains(@class, 'cust-row-name')])[1]").click()
        time.sleep(2)
        wait.until(EC.visibility_of_element_located((By.XPATH, f"//h3[text()='{names[0]}']")))
        print("PASS: Customer details opened")

    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'CRM')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='CRM sections']")))
    print("PASS: CRM opened")

    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Dashboard')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//section[@aria-label='Pharmacist dashboard']")))
    print("PASS: Dashboard restored")

    driver.refresh()
    time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    assert not driver.find_elements(By.ID, "username")
    print("PASS: Refresh successful")

    # Real logout UI: header Account menu -> Sign Out (verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@aria-label='Account menu']"))).click()
    time.sleep(1)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//div[@role='menu']//button[@role='menuitem' and contains(., 'Sign Out')]"))).click()
    time.sleep(3)
    wait.until(EC.presence_of_element_located((By.ID, "username")))
    print("PASS: Logout successful")
    print("PASS: Login page displayed")
    print("PASS: Full basic user flow completed")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("60_full_flow_FAIL.png")
finally:
    driver.quit()